In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score

# Custom modules
from src.vgg11bn import VGG11BN
from src.utils import (
    ImageDataset, set_seed, print_framed_metrics, compute_npz_mean_std, 
    enable_dropout, mc_dropout_inference, load_problem2_data
)
from src.train import train_model
from src.evaluation import (
    classification_summary, plot_confusion_matrix, plot_per_class_accuracy,
    plot_per_class_metrics, plot_multiclass_roc, plot_calibration_curve,
    plot_entropy_hist, plot_predictions
)

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Set seed for reproducibility and determine device
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Enable cuDNN benchmarking for faster training
torch.backends.cudnn.benchmark = True

### Data loading and preprocessing


In [ ]:
# Load the dataset
print("Loading datasets...")
train_images, train_labels, val_images, val_labels = load_problem2_data()

# Combine training and validation data for better split
combined_images = np.concatenate((train_images, val_images), axis=0)
combined_labels = np.concatenate((train_labels, val_labels), axis=0)

# Create a new train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    combined_images, combined_labels, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} images")
print(f"Validation set: {len(X_val)} images")

# Compute dataset statistics for normalization
ds_mean, ds_std = compute_npz_mean_std(combined_images)
print(f"Dataset mean: {ds_mean}, std: {ds_std}")

# Define data augmentation for training
train_transform = v2.Compose([
    v2.ToPILImage(),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(10),
    v2.RandomResizedCrop(96, scale=(0.8, 1.0)),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=ds_mean.tolist(), std=ds_std.tolist())
])

# Define validation transforms (no augmentation)
val_transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=ds_mean.tolist(), std=ds_std.tolist())
])

# Create datasets
train_dataset = ImageDataset(X_train, y_train, transform=train_transform)
val_dataset = ImageDataset(X_val, y_val, transform=val_transform)

# Define class names for visualization
class_names = ['Plane', 'Ship', 'Truck']

### Task 2a & 2c: VGG11 w/ BatchNorm (and Dropout)


In [ ]:
# 2a: VGG11 with BatchNorm and No Dropout
def train_vgg11bn(with_dropout=False, batch_size=32):
    print(f"\n\n=== Training VGG11BN {'with' if with_dropout else 'without'} Dropout (BS: {batch_size}) ===")
    
    # Create dataloaders
    num_workers = min(os.cpu_count() - 1, 4) if os.cpu_count() > 1 else 0
    dataloaders = {
        'train': DataLoader(
            train_dataset, 
            batch_size=batch_size,
            shuffle=True, 
            pin_memory=True,
            num_workers=num_workers, 
            persistent_workers=True if num_workers > 0 else False,
            prefetch_factor=4 if num_workers > 0 else None
        ),
        'val': DataLoader(
            val_dataset, 
            batch_size=batch_size,
            shuffle=False, 
            pin_memory=True,
            num_workers=num_workers, 
            persistent_workers=True if num_workers > 0 else False
        )
    }
    
    # Create the model
    model = VGG11BN(num_classes=len(class_names), dropout=with_dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer: SGD with Nesterov momentum
    optimizer = optim.SGD(
        model.parameters(),
        lr=1e-2,  # Higher learning rate for SGD
        momentum=0.9,
        nesterov=True,
        weight_decay=5e-4
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.35, 
        patience=10,
        verbose=True
    )
    
    # Train the model
    model, model_name = train_model(
        model=model,
        dataloaders=dataloaders,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=100,
        model_name=f"2{'c' if with_dropout else 'a'}_VGG11BN_dropout{with_dropout}_bs{batch_size}",
        patience=15
    )
    
    # Evaluate the model
    evaluate_model(model, dataloaders['val'], criterion, f"2{'c' if with_dropout else 'a'}_dropout{with_dropout}_bs{batch_size}")
    
    return model, dataloaders

def evaluate_model(model, val_loader, criterion, name_suffix):
    """Comprehensive model evaluation with all the metrics"""
    print("\n=== Model Evaluation ===")
    
    # Basic classification metrics
    val_loss, val_acc, y_true, y_pred = classification_summary(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
        class_names=class_names
    )
    
    print_framed_metrics(val_loss, val_acc)
    
    # Per-class accuracy
    plot_per_class_accuracy(
        y_true, y_pred, 
        class_names=class_names,
        save_path=f"plots/{name_suffix}_per_class_acc.png"
    )
    
    # Per-class metrics (precision, recall, F1)
    plot_per_class_metrics(
        y_true, y_pred, 
        class_names=class_names,
        save_path=f"plots/{name_suffix}_per_class_metrics.png"
    )
    
    # Confusion matrix
    plot_confusion_matrix(
        model,
        val_loader,
        device,
        class_names=class_names,
        save_path=f"plots/{name_suffix}_confusion_matrix.png"
    )
    
    # ROC curves
    plot_multiclass_roc(
        model,
        val_loader,
        device,
        num_classes=len(class_names),
        class_names=class_names,
        model_name=name_suffix,
        save_path=f"plots/{name_suffix}_roc_curves.png"
    )
    
    # Calibration curve
    plot_calibration_curve(
        model, 
        val_loader, 
        device, 
        num_classes=len(class_names),
        class_names=class_names,
        save_path=f"plots/{name_suffix}_calibration_curve.png"
    )
    
    # Visualize predictions
    plot_predictions(
        model, 
        val_loader, 
        class_names, 
        device, 
        num_samples=16,
        save_path=f"plots/{name_suffix}_predictions.png"
    )
    
    return val_loss, val_acc, y_true, y_pred

#### Training models for 2a and 2c

In [ ]:
# Task 2a: Train VGG11BN without Dropout
model_2a, dataloaders_2a = train_vgg11bn(with_dropout=False, batch_size=32)

# Task 2c: Train VGG11BN with Dropout
model_2c, dataloaders_2c = train_vgg11bn(with_dropout=True, batch_size=32)

### 2d : Uncertainity estimation

In [ ]:
# Load the new dataset with known/unknown labels
print("\n\n=== Task 2d: Uncertainty Estimation ===")
eval_data = np.load('data/problem2/new_evaluation_data_with_labels.npz')
eval_images = eval_data['a']
eval_labels = eval_data['b']

# Load unlabeled data
unlabeled_data = np.load('data/problem2/new_evaluation_data_without_labels.npz')
unlabeled_images = unlabeled_data['a']

print(f"Evaluation dataset: {eval_images.shape} images, {len(np.unique(eval_labels))} classes")
print(f"Unlabeled dataset: {unlabeled_images.shape} images")

# Split data into known (planes, ships, trucks) and unknown (birds)
known_images = eval_images[eval_labels != 3]
unknown_images = eval_images[eval_labels == 3]
print(f"Known class images: {known_images.shape}")
print(f"Unknown class images (birds): {unknown_images.shape}")

# Create DataLoaders
known_dataset = ImageDataset(known_images, transform=val_transform)
unknown_dataset = ImageDataset(unknown_images, transform=val_transform)
known_loader = DataLoader(known_dataset, batch_size=32, shuffle=False)
unknown_loader = DataLoader(unknown_dataset, batch_size=32, shuffle=False)

# Use the model with dropout for uncertainty estimation
model = model_2c  # The model trained with dropout
model.eval()
enable_dropout(model)  # Keep dropout active for MC Dropout

# Calculate entropy for known and unknown classes
entropies_known = []
entropies_unknown = []

print("Processing known classes...")
for batch in known_loader:
    for img in batch:
        mean_probs, entropy = mc_dropout_inference(model, img, T=10, device=device)
        entropies_known.append(entropy)

print("Processing unknown class (birds)...")
for batch in unknown_loader:
    for img in batch:
        mean_probs, entropy = mc_dropout_inference(model, img, T=10, device=device)
        entropies_unknown.append(entropy)

# Convert to numpy arrays
entropies_known = np.array(entropies_known)
entropies_unknown = np.array(entropies_unknown)

# Plot entropy distributions
plot_entropy_hist(
    entropies_known, 
    entropies_unknown,
    save_path="plots/entropy_distributions.png"
)

# Calculate and print statistics
mean_known = np.mean(entropies_known)
mean_unknown = np.mean(entropies_unknown)
std_known = np.std(entropies_known)
std_unknown = np.std(entropies_unknown)

print("\nEntropy Statistics:")
print(f"Known Classes - Mean: {mean_known:.4f}, Std: {std_known:.4f}")
print(f"Unknown Class - Mean: {mean_unknown:.4f}, Std: {std_unknown:.4f}")
print(f"Mean Difference: {mean_unknown - mean_known:.4f}")

### 2e: Threshold Selection

In [ ]:
print("\n\n=== Task 2e: Threshold Selection ===")

# Calculate midpoint threshold
midpoint_threshold = (mean_known + mean_unknown) / 2
print(f"Midpoint Threshold: {midpoint_threshold:.4f}")

# Calculate optimal threshold using ROC curve and Youden's J statistic
# Combine labels (0 for known, 1 for unknown) and entropies
labels_combined = np.concatenate([
    np.zeros(len(entropies_known)),  # Known samples = 0
    np.ones(len(entropies_unknown))   # Unknown samples = 1
])

# Combine entropy values
entropies_combined = np.concatenate([
    entropies_known,
    entropies_unknown
])

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(labels_combined, entropies_combined)

# Find optimal threshold using Youden's J statistic (maximizing TPR-FPR)
J = tpr - fpr
best_idx = np.argmax(J)
optimal_threshold = thresholds[best_idx]

# Calculate AUC
auc = roc_auc_score(labels_combined, entropies_combined)
print(f"ROC AUC: {auc:.4f}")
print(f"Optimal Threshold (Youden's J): {optimal_threshold:.4f}")

# Plot ROC curve
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.scatter(fpr[best_idx], tpr[best_idx], color='red', 
            label=f"Optimal Threshold = {optimal_threshold:.4f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Unknown Class Detection")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("plots/roc_curve_entropy.png")
plt.show()

# Plot TPR and FPR vs threshold
plt.figure(figsize=(10, 7))
plt.plot(thresholds, tpr, label="True Positive Rate (TPR)")
plt.plot(thresholds, fpr, label="False Positive Rate (FPR)")
plt.axvline(optimal_threshold, color='red', linestyle='--', 
            label=f"Optimal Threshold = {optimal_threshold:.4f}")
plt.axvline(midpoint_threshold, color='green', linestyle='--', 
            label=f"Midpoint Threshold = {midpoint_threshold:.4f}")
plt.xlabel("Entropy Threshold")
plt.ylabel("Rate")
plt.title("TPR/FPR vs Entropy Threshold")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("plots/tpr_fpr_vs_threshold.png")
plt.show()

# Evaluate thresholds on the labeled data
# 1. With the optimal threshold
detected_birds_optimal = np.sum(entropies_unknown >= optimal_threshold)
removed_knowns_optimal = np.sum(entropies_known >= optimal_threshold)
bird_detection_rate_optimal = detected_birds_optimal / len(entropies_unknown) * 100
false_removal_rate_optimal = removed_knowns_optimal / len(entropies_known) * 100

print("\nResults with Optimal Threshold:")
print(f"Detected Birds: {detected_birds_optimal}/{len(entropies_unknown)} ({bird_detection_rate_optimal:.2f}%)")
print(f"Removed Known Samples: {removed_knowns_optimal}/{len(entropies_known)} ({false_removal_rate_optimal:.2f}%)")

# 2. With the midpoint threshold
detected_birds_midpoint = np.sum(entropies_unknown >= midpoint_threshold)
removed_knowns_midpoint = np.sum(entropies_known >= midpoint_threshold)
bird_detection_rate_midpoint = detected_birds_midpoint / len(entropies_unknown) * 100
false_removal_rate_midpoint = removed_knowns_midpoint / len(entropies_known) * 100

print("\nResults with Midpoint Threshold:")
print(f"Detected Birds: {detected_birds_midpoint}/{len(entropies_unknown)} ({bird_detection_rate_midpoint:.2f}%)")
print(f"Removed Known Samples: {removed_knowns_midpoint}/{len(entropies_known)} ({false_removal_rate_midpoint:.2f}%)")

# Choose the best threshold based on a balance of detection and false removal
print("\nTradeoff Analysis:")
optimal_f1 = (2 * detected_birds_optimal * (len(entropies_known) - removed_knowns_optimal)) / \
            (2 * detected_birds_optimal * (len(entropies_known) - removed_knowns_optimal) + 
             (len(entropies_unknown) - detected_birds_optimal) * removed_knowns_optimal)

midpoint_f1 = (2 * detected_birds_midpoint * (len(entropies_known) - removed_knowns_midpoint)) / \
              (2 * detected_birds_midpoint * (len(entropies_known) - removed_knowns_midpoint) + 
               (len(entropies_unknown) - detected_birds_midpoint) * removed_knowns_midpoint)

print(f"Optimal Threshold F1: {optimal_f1:.4f}")
print(f"Midpoint Threshold F1: {midpoint_f1:.4f}")

# Choose final threshold
final_threshold = optimal_threshold if optimal_f1 > midpoint_f1 else midpoint_threshold
print(f"\nSelected Threshold: {final_threshold:.4f}")

### 2f: Unknown class filtering

In [ ]:
print("\n\n=== Task 2f: Filtering Unknown Classes ===")

# Prepare model for MC Dropout inference
model.eval()
enable_dropout(model)

# Create dataset and loader for unlabeled data
unlabeled_dataset = ImageDataset(unlabeled_images, transform=val_transform)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=1, shuffle=False)

print("Running inference on unlabeled dataset...")
binary_predictions = []

for i, batch in enumerate(unlabeled_loader):
    if i % 100 == 0:
        print(f"Processing image {i}/{len(unlabeled_loader)}")
        
    img = batch[0] if isinstance(batch, tuple) else batch
    img = img.squeeze(0)  # (1, C, H, W) → (C, H, W)

    mean_probs, entropy = mc_dropout_inference(model, img, T=10, device=device)
    prediction = 1 if entropy >= final_threshold else 0  # 1 = unknown (bird), 0 = known
    binary_predictions.append(prediction)

# Convert to numpy array
binary_predictions = np.array(binary_predictions)

# Save predictions
output_file = 'binary_predictions.npz'
np.savez(output_file, predictions=binary_predictions)
print(f"Saved binary predictions to {output_file}")

# Summary statistics
unknown_count = np.sum(binary_predictions)
known_count = len(binary_predictions) - unknown_count
print(f"Predicted {unknown_count} unknown samples (birds)")
print(f"Predicted {known_count} known samples (planes, ships, trucks)")
print(f"Unknown ratio: {unknown_count / len(binary_predictions) * 100:.2f}%")

In [ ]:
# Save the best model path for future reference
with open("best_model_info.txt", "w") as f:
    f.write(f"Threshold: {final_threshold}\n")
    f.write(f"Bird Detection Rate: {bird_detection_rate_optimal:.2f}%\n")
    f.write(f"False Removal Rate: {false_removal_rate_optimal:.2f}%\n")

# <center>**Summary and Conclusion**</center>


#### <center>**Task 2a: VGG-11 with Batch Normalization**</center>

VGG-11 with Batch Normalization was chosen for this image classification task because:
1. It's moderately complex but not excessively deep for the 96x96 image size
2. Batch normalization helps stabilize training and accelerate convergence
3. The architecture has shown effectiveness on similar image classification tasks
4. The convolutional patterns can capture the visual features of planes, ships, and trucks
-----------------------------------------
#### <center>**Task 2b: Dropout Explanation**</center>

Dropout is a regularization technique that randomly sets a fraction of inputs to zero during training. 
It helps prevent overfitting by:
1. Forcing the network to learn redundant representations
2. Creating an implicit ensemble of different network architectures
3. Reducing co-adaptation between neurons
4. Acting as a form of data augmentation at the feature level
---------------------------
#### <center>**Task 2c: VGG-11 with Dropout Results**</center>

Adding dropout with rate 0.5 after the two 4096-unit fully connected layers:
1. Decreased training speed slightly but improved generalization
2. Reduced overfitting compared to the non-dropout model
3. Allowed the model to express uncertainty better (important for the OOD detection)
4. Combined with ReduceLROnPlateau scheduler, achieved our best validation accuracy
-----------------------------------
#### <center>**Task 2d-2f: Uncertainty Estimation and Unknown Class Detection**</center>

Using Monte Carlo Dropout for uncertainty estimation:
1. We performed 10 forward passes with dropout enabled at inference time
2. Calculated entropy of the mean softmax outputs as our uncertainty measure
3. Found clear separation between entropies of known vs unknown classes
4. Selected optimal threshold that maximizes unknown detection while minimizing false removals
5. Applied this threshold to filter out bird images from the unlabeled dataset
-------------------------------------------------------------



